# P07 Производная и проверка градиента

**Эталон решения**. Полное объяснение: [локальный урок P07](../../course/pilot/P07.md). Этот файл содержит исполняемый практикум.

Начните с [порядка самостоятельной работы](../../course/self-study.md). Локальный учебник открывается файлом START_HERE.html в корне курса. Если ноутбук скопирован в submissions и относительная ссылка не открывается, найдите урок по пути course/pilot/P07.md от корня репозитория.

## Цель

Объяснить скорость изменения и сопоставить аналитическую производную с численной.

**Перед началом:** Функции, разность, деление; P02 об ошибке. Частная производная рассматривает один параметр при фиксированных остальных.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

Для f(x)=x² производная равна 2x. В точке x=2 она равна 4: для достаточно малого изменения входа изменение функции приблизительно в четыре раза больше. Центральная разность использует две близкие точки и приближает наклон; слишком маленький шаг может усилить ошибки округления.

### Сравнить формулу и конечную разность

In [ ]:
f = lambda value: value**2
numeric = central_difference(f, 2., epsilon=1e-5)
print("Аналитически:", 4, "численно:", numeric)
assert np.isclose(numeric, 4, atol=1e-8)

### Показать касательную

In [ ]:
grid = np.linspace(0.5, 3.5, 120)
fig, ax = plt.subplots()
ax.plot(grid, grid**2, color="#1f5a99", label="f(x)=x²")
ax.plot(grid, 4 + 4*(grid-2), color="#333333", linestyle="--", label="Касательная при x=2")
ax.scatter([2], [4], color="#b56c19")
ax.set(xlabel="x", ylabel="f(x)", title="Производная как наклон")
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "P07_derivative.png", dpi=130); plt.show()

### Проверить частные производные ошибки

In [ ]:
x = np.array([0., 1., 2.]); y = np.array([1., 3., 5.])
dw, db = linear_gradient(x, y, 0, 0)
numeric_dw = central_difference(lambda w: mse(y, predict_line(x, w, 0)), 0)
numeric_db = central_difference(lambda b: mse(y, predict_line(x, 0, b)), 0)
np.testing.assert_allclose([dw, db], [numeric_dw, numeric_db], atol=1e-7)
print("dL/dw:", dw, "dL/db:", db)

## Практика с поддержкой

Объясните, что фиксировано при вычислении производной по w. Проверьте f'(3) для f(x)=x².

## Самостоятельная задача

Для g(t)=(t+1)² выведите производную и проверьте её в t=2 центральной разностью. Объясните знак. Для одного примера x=2, y=5 при w=1, b=0 рассчитайте ошибку e=wx+b−y и производные квадрата ошибки по w и b.

Сначала сохраните собственную попытку. Затем откройте подсказки в локальном уроке и сверяйтесь с эталоном; отмечайте, какая помощь потребовалась. После полного разбора выполните новый вариант без подсказок. При желании можно передать работу на разбор.

### Эталон

Открывайте после собственной попытки. После разбора понадобится новый вариант для проверки самостоятельности.

In [ ]:
g = lambda t: (t+1)**2
assert np.isclose(central_difference(g, 2), 6, atol=1e-8)
assert 2*(2+1) == 6
error = 1*2 + 0 - 5
assert error == -3
np.testing.assert_allclose(linear_gradient([2], [5], 1, 0), [-12, -6])
print("g'(2)=6: малое увеличение t увеличивает g. e=-3, dL/dw=-12, dL/db=-6.")

## Проверка

Производная не перепутана со значением функции; знак объяснён; численная проверка не используется как точное равенство для всех epsilon.

Успешный Run All подтверждает запуск подготовленных примеров. Для самопроверки сравните свою работу с критериями и эталоном, объясните шаги своими словами и решите новый вариант. Запишите результат в progress/progress.md относительно корня курса. Самооценка не является независимым подтверждением навыка.

## Следующий шаг

Для h(t)=(t−3)² вычислите h'(4) и объясните, в какую сторону уменьшать h. Затем сравните несколько epsilon.

[Основной источник](https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.